In [23]:
import pandas as pd
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import numpy as np

In [24]:
# Colonnes du dataset  récupéré depuis le readme.md du dataset
COLONNES = [
    "id",                   
    "label",                
    "statement",            
    "subject",              
    "speaker",              
    "job_title",            
    "state",                
    "party",                
    "barely_true_counts",   
    "false_counts",         
    "half_true_counts",     
    "mostly_true_counts",   
    "pants_on_fire_counts",
    "context"               
]

# Chargement des données 
train = pd.read_csv("data/raw/train.tsv", sep="\t", header=None, names=COLONNES)
valid = pd.read_csv("data/raw/valid.tsv", sep="\t", header=None, names=COLONNES)
test  = pd.read_csv("data/raw/test.tsv",  sep="\t", header=None, names=COLONNES)

# Vérifiation
print("Chargement réussi !")
print(f"   Train : {train.shape[0]} lignes, {train.shape[1]} colonnes")
print(f"   Valid : {valid.shape[0]} lignes, {valid.shape[1]} colonnes")
print(f"   Test  : {test.shape[0]} lignes, {test.shape[1]} colonnes")

Chargement réussi !
   Train : 10240 lignes, 14 colonnes
   Valid : 1284 lignes, 14 colonnes
   Test  : 1267 lignes, 14 colonnes


VÉRIFICATION DES COLONNES

In [25]:
print("Colonnes Train :", train.columns.tolist())
print("Colonnes Valid :", valid.columns.tolist())
print("Colonnes Test :", test.columns.tolist())

# Vérification automatiqu
if train.columns.tolist() == valid.columns.tolist() == test.columns.tolist():
    print("\n Les 3 datasets ont exactement les mêmes colonnes !")
else:
    print("\n Attention, les colonnes sont différentes !")

Colonnes Train : ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']
Colonnes Valid : ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']
Colonnes Test : ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']

 Les 3 datasets ont exactement les mêmes colonnes !


Apercu des données

In [26]:
print("Apercu des 5 premières lignes du dataset train :")
train.head(5)

Apercu des 5 premières lignes du dataset train :


,id,label,statement,subject,speaker,job_title,state,party,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN


 VALEURS MANQUANTES SUR LES 3 DATASETS




In [27]:
for nom, df in [("Train", train), ("Valid", valid), ("Test", test)]:
    print(f"\n### {nom}")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    result = pd.DataFrame({
        "Valeurs manquantes": missing,
        "Pourcentage (%)": missing_pct
    })
    
    display(result[result["Valeurs manquantes"] > 0])


### Train


,Valeurs manquantes,Pourcentage (%)
subject,2,0.02
speaker,2,0.02
job_title,2898,28.30
state,2210,21.58
party,2,0.02
barely_true_counts,2,0.02
false_counts,2,0.02
half_true_counts,2,0.02
mostly_true_counts,2,0.02
pants_on_fire_counts,2,0.02



### Valid


,Valeurs manquantes,Pourcentage (%)
job_title,345,26.87
state,279,21.73
context,12,0.93



### Test


,Valeurs manquantes,Pourcentage (%)
job_title,325,25.65
state,262,20.68
context,17,1.34


Après analyse des 3 datasets (train, valid, test), on identifie les valeurs manquantes suivantes

**`job_title` et `state` (≈25%) → on ne fait rien**
Ces colonnes ne seront pas utilisées dans notre pipeline de classification.
Le texte principal (`statement`) suffit pour entraîner le modèle.
Supprimer ces lignes ferait perdre 25% des données inutilement.

**`context`, `speaker`, `party`, `subject` (< 1%) → on remplace par une valeur neutre**
Ces colonnes sont utiles notamment pour l'analyse des biais (par parti, par speaker).
On remplace les manquants par `"unknown"` ou `""` pour ne perdre aucune ligne.
La règle d'or : on ne supprime une ligne que si elle est complètement inutilisable.

**`statement` et `label` → aucune action nécessaire**
Ces deux colonnes sont les plus importantes du projet :
`statement` est le texte qu'on classifie, `label` est ce qu'on cherche à prédire.
Heureusement elles ne contiennent aucune valeur manquante sur les 3 datasets.

Nettoyage des données

In [28]:
for df in [train, valid, test]:
    df["context"] = df["context"].fillna("")
    df["speaker"] = df["speaker"].fillna("unknown")
    df["party"]   = df["party"].fillna("unknown")
    df["subject"] = df["subject"].fillna("")

print(" Nettoyage terminé ")
print(f"   Train : {len(train)} lignes")
print(f"   Valid : {len(valid)} lignes")
print(f"   Test  : {len(test)} lignes")

# Vérification : plus aucune valeur manquante sur les colonnes utiles
cols_utiles = ["label", "statement", "speaker", "party", "context", "subject"]
print("\n### Vérification finale sur les colonnes utiles")
for nom, df in [("Train", train), ("Valid", valid), ("Test", test)]:
    nb = df[cols_utiles].isnull().sum().sum()
    print(f"   {nom} : {nb} valeurs manquantes restantes")

 Nettoyage terminé 
   Train : 10240 lignes
   Valid : 1284 lignes
   Test  : 1267 lignes

### Vérification finale sur les colonnes utiles
   Train : 0 valeurs manquantes restantes
   Valid : 0 valeurs manquantes restantes
   Test : 0 valeurs manquantes restantes


## Distribution des labels

On analyse ici la répartition des 6 labels de véracité dans le train set.
C'est une étape cruciale pour détecter un éventuel déséquilibre des classes
qui pourrait biaiser notre modèle.

In [29]:
ORDRE_LABELS = [
    "pants-fire",
    "false",
    "barely-true",
    "half-true",
    "mostly-true",
    "true"
]

COULEURS = [
    "#d73027", 
    "#f46d43",  
    "#fdae61", 
    "#fee08b",  
    "#a6d96a", 
    "#1a9850"   
]

counts = train["label"].value_counts().reindex(ORDRE_LABELS)

fig = px.bar(
    x=counts.index,
    y=counts.values,
    color=counts.index,
    color_discrete_sequence=COULEURS,
    title="Distribution des labels — Train set",
    labels={"x": "Label de véracité", "y": "Nombre de déclarations"},
    text=counts.values
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

# Résumé chiffré
print("\n### Résumé")
for label, count in counts.items():
    pct = count / len(train) * 100
    print(f"   {label:<20} : {count} ({pct:.1f}%)")


### Résumé
   pants-fire           : 839 (8.2%)
   false                : 1995 (19.5%)
   barely-true          : 1654 (16.2%)
   half-true            : 2114 (20.6%)
   mostly-true          : 1962 (19.2%)
   true                 : 1676 (16.4%)


**Bonne nouvelle** : la majorité des classes sont relativement équilibrées
entre 16% et 20%, ce qui est favorable pour l'entraînement du modèle.

**Point d'attention** : `pants-fire` est sous-représenté avec seulement 8.2%.
C'est deux fois moins que les autres classes. Le modèle aura donc
moins d'exemples pour apprendre à détecter les mensonges flagrants.

On devra gérer ce léger déséquilibre avec l'une de ces stratégies :
- `class_weight="balanced"` dans scikit-learn
- Regroupement des labels en binaire (fake / true)
- Oversampling de la classe minoritaire


A voir après 

##  Analyse des speakers et des partis politiques

On analyse ici qui parle le plus dans le dataset et
quel est le lien entre le parti politique et la véracité des déclarations.
C'est aussi la base de notre analyse des biais plus tard.

In [30]:
top_speakers = train["speaker"].value_counts().head(10).reset_index()
top_speakers.columns = ["speaker", "count"]

fig = px.bar(
    top_speakers,
    x="count",
    y="speaker",
    orientation="h",
    title="Top 10 speakers les plus présents",
    labels={"count": "Nombre de déclarations", "speaker": "Speaker"},
    color="count",
    color_continuous_scale="Blues",
    text="count"
)

fig.update_traces(textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
fig.show()

Barack Obama est de loin le speaker le plus représenté avec 488 déclarations,
suivi de Donald Trump (273) et Hillary Clinton (239).
On note aussi "chain-email" qui représente des emails viraux anonymes.

Biais potentiel : le modèle risque de sur-apprendre
les patterns propres à Obama vu sa surreprésentation dans le dataset.

In [31]:

top_partis = train["party"].value_counts().head(8).reset_index()
top_partis.columns = ["party", "count"]

fig = px.bar(
    top_partis,
    x="party",
    y="count",
    title="Distribution des déclarations par parti politique",
    labels={"count": "Nombre de déclarations", "party": "Parti"},
    color="count",
    color_continuous_scale="Purples",
    text="count"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

Le dataset est dominé par deux partis :
- Republican : 4497 déclarations (44%)
- Democrat : 3336 déclarations (33%)
- None : 1744 déclarations (17%)

Les autres partis (independent, libertarian, activist) sont très minoritaires.

Biais potentiel : le modèle sera beaucoup mieux calibré
pour les déclarations républicaines et démocrates que pour les autres.

In [32]:

# Garder uniquement les 5 partis les plus représentés
top_5_partis = train["party"].value_counts().head(5).index

df_partis = train[train["party"].isin(top_5_partis)]

# Calculer la proportion de chaque label par parti
df_grouped = (
    df_partis
    .groupby(["party", "label"])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    df_grouped,
    x="party",
    y="count",
    color="label",
    title="Véracité des déclarations par parti politique (top 5)",
    labels={"count": "Nombre de déclarations", "party": "Parti", "label": "Label"},
    category_orders={"label": ORDRE_LABELS},
    color_discrete_sequence=COULEURS,
    barmode="group"
)

fig.show()

On observe une différence notable entre les partis :
- Les républicains ont plus de `pants-fire` et `false` que les démocrates
- Les démocrates ont proportionnellement plus de `mostly-true` 

Attention : cela ne veut pas dire qu'un parti ment plus que l'autre.
Cela peut refléter un biais dans les sources annotées par PolitiFact,
qui a peut-être vérifié davantage de déclarations républicaines controversées.

## Analyse de la longueur des déclarations

On analyse ici la longueur des déclarations en nombre de mots.
C'est important car les textes courts posent un défi particulier
pour les modèles NLP qui ont besoin de contexte pour bien classifier.

In [33]:

# Calcul du nombre de mots
train["nb_mots"] = train["statement"].str.split().str.len()

print("### Statistiques de longueur")
print(train["nb_mots"].describe().round(2))

# Histogramme
fig = px.histogram(
    train,
    x="nb_mots",
    nbins=50,
    title="Distribution de la longueur des déclarations",
    labels={"nb_mots": "Nombre de mots", "count": "Fréquence"},
    color_discrete_sequence=["#4C72B0"]
)

fig.add_vline(
    x=train["nb_mots"].mean(),
    line_dash="dash",
    line_color="red",
    annotation_text=f"Moyenne : {train['nb_mots'].mean():.1f} mots",
    annotation_position="top right"
)

fig.show()

### Statistiques de longueur
count    10240.00
mean        18.01
std          9.66
min          2.00
25%         12.00
50%         17.00
75%         22.00
max        467.00
Name: nb_mots, dtype: float64


es déclarations sont en moyenne très courtes : seulement 18 mots.
75% des déclarations font moins de 22 mots ce qui confirme
que le dataset est composé majoritairement de phrases courtes.

Défi pour la modélisation : les textes courts contiennent
peu de contexte, ce qui rend la classification plus difficile.
C'est une des raisons pour lesquelles BERT sera plus performant
que TF-IDF — il capte mieux le sens avec peu de mots.

Valeur maximale de 467 mots : il existe quelques outliers
très longs qui pourraient perturber certains modèles.
On les gardera mais on en tiendra compte dans l'analyse des erreurs.

In [34]:
moyenne_par_label = (
    train.groupby("label")["nb_mots"]
    .mean()
    .reindex(ORDRE_LABELS)
    .round(1)
    .reset_index()
)
moyenne_par_label.columns = ["label", "moyenne_mots"]

fig = px.bar(
    moyenne_par_label,
    x="label",
    y="moyenne_mots",
    color="label",
    title="Longueur moyenne des déclarations par label",
    labels={"moyenne_mots": "Nombre de mots moyen", "label": "Label"},
    color_discrete_sequence=COULEURS,
    text="moyenne_mots"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

Les différences de longueur entre les labels sont très faibles
(entre 17 et 18.8 mots en moyenne).

On ne peut donc pas utiliser la longueur comme feature
discriminante pour détecter les fake news.

Cela confirme que le modèle devra s'appuyer uniquement
sur le contenu sémantique des mots et non sur la longueur.

## Historique de véracité par speaker

Le LIAR Dataset contient 5 colonnes d'historique : le nombre de fois que chaque speaker
a reçu chaque label sur PolitiFact **avant** cette déclaration.

On analyse ici si cet historique est corrélé avec le label actuel de la déclaration.
C'est une feature potentiellement très utile — et une source de biais à documenter.

In [35]:
ORDRE_LABELS = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]
COULEURS     = ["#d73027", "#f46d43", "#fdae61", "#fee08b", "#a6d96a", "#1a9850"]

# Colonnes d'historique disponibles dans le dataset
HISTORY_COLS = [
    "pants_on_fire_counts",
    "false_counts",
    "barely_true_counts",
    "half_true_counts",
    "mostly_true_counts",
]

# Calcul du score de crédibilité : ratio (déclarations vraies) / (total historique)
# Plus ce score est élevé, plus le speaker a un historique fiable
train["total_counts"] = train[HISTORY_COLS].sum(axis=1)
train["credibility_score"] = (
    (train["mostly_true_counts"] + train["half_true_counts"])
    / train["total_counts"].replace(0, 1)  # évite la division par zéro
).round(3)

# Moyenne du score de crédibilité par label
cred_by_label = (
    train.groupby("label")["credibility_score"]
    .mean()
    .reindex(ORDRE_LABELS)
    .round(3)
    .reset_index()
)
cred_by_label.columns = ["label", "credibility_score_moyen"]

fig = px.bar(
    cred_by_label,
    x="label",
    y="credibility_score_moyen",
    color="label",
    color_discrete_sequence=COULEURS,
    title="Score de crédibilité historique moyen par label",
    labels={"credibility_score_moyen": "Score moyen (mostly-true + half-true) / total", "label": "Label"},
    text="credibility_score_moyen"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

# Résumé chiffré
print("### Score de crédibilité moyen par label")
for _, row in cred_by_label.iterrows():
    print(f"   {row['label']:<20} : {row['credibility_score_moyen']:.3f}")

### Score de crédibilité moyen par label
   pants-fire           : 0.245
   false                : 0.315
   barely-true          : 0.339
   half-true            : 0.649
   mostly-true          : 0.674
   true                 : 0.441


In [36]:
# Heatmap : moyenne de chaque colonne d'historique par label
heatmap_data = (
    train.groupby("label")[HISTORY_COLS]
    .mean()
    .reindex(ORDRE_LABELS)
    .round(1)
)

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=[c.replace("_counts", "").replace("_", "-") for c in HISTORY_COLS],
    y=ORDRE_LABELS,
    colorscale="RdYlGn",
    text=heatmap_data.values,
    texttemplate="%{text}",
    showscale=True
))

fig.update_layout(
    title="Historique moyen de véracité par label actuel (train set)",
    xaxis_title="Type de déclaration dans l'historique",
    yaxis_title="Label actuel de la déclaration"
)
fig.show()

**Lecture de la heatmap** : les speakers dont le label actuel est `pants-fire` ou `false`
ont en moyenne un historique avec plus de `false_counts` et `pants_on_fire_counts`.
À l'inverse, les speakers labellisés `true` ou `mostly-true` ont un historique
proportionnellement plus riche en déclarations vraies.

**Biais potentiel** : si on utilise cet historique comme feature, le modèle va apprendre
à juger le *speaker* autant que la *déclaration*. Un speaker habituellement menteur
pourrait se voir attribuer un label `false` même pour une déclaration vraie.
C'est un biais éthique important à signaler dans le rapport.

**Décision** : on peut intégrer le `credibility_score` comme feature additionnelle
dans les modèles classiques (Logistic Regression, Random Forest), mais en étant
conscient de ce biais.

## Analyse des sujets (topics)

La colonne `subject` contient les thèmes des déclarations (santé, économie, immigration…).
On analyse ici quels sujets sont les plus présents et si certains sujets
sont plus associés aux fake news que d'autres.

In [37]:
# Chaque ligne peut avoir plusieurs sujets séparés par des virgules
# On les éclate pour compter chaque sujet individuellement
subjects_exploded = (
    train["subject"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

top_subjects = subjects_exploded.value_counts().head(15).reset_index()
top_subjects.columns = ["subject", "count"]

fig = px.bar(
    top_subjects,
    x="count",
    y="subject",
    orientation="h",
    title="Top 15 sujets les plus fréquents dans le train set",
    labels={"count": "Nombre de déclarations", "subject": "Sujet"},
    color="count",
    color_continuous_scale="Blues",
    text="count"
)

fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    showlegend=False
)
fig.show()

print("### Top 10 sujets")
for _, row in top_subjects.head(10).iterrows():
    print(f"   {row['subject']:<30} : {row['count']} déclarations")

### Top 10 sujets
   economy                        : 1162 déclarations
   health-care                    : 1128 déclarations
   taxes                          : 994 déclarations
   federal-budget                 : 744 déclarations
   education                      : 728 déclarations
   jobs                           : 711 déclarations
   state-budget                   : 696 déclarations
   candidates-biography           : 653 déclarations
   elections                      : 607 déclarations
   immigration                    : 532 déclarations


In [38]:
# Taux de fake news par sujet (pants-fire + false) pour les 12 sujets principaux
TOP_SUBJECTS = top_subjects["subject"].head(12).tolist()

# Associer chaque déclaration à son premier sujet principal
train["subject_principal"] = (
    train["subject"]
    .str.split(",")
    .str[0]
    .str.strip()
)

# Filtrer sur les sujets principaux
df_top = train[train["subject_principal"].isin(TOP_SUBJECTS)].copy()

# Calcul du taux de fake (pants-fire + false) par sujet
df_top["is_fake"] = df_top["label"].isin(["pants-fire", "false"]).astype(int)

fake_rate = (
    df_top.groupby("subject_principal")["is_fake"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
fake_rate.columns = ["subject", "taux_fake"]
fake_rate["taux_fake_pct"] = (fake_rate["taux_fake"] * 100).round(1)

fig = px.bar(
    fake_rate,
    x="subject",
    y="taux_fake_pct",
    title="Taux de déclarations fake (pants-fire + false) par sujet principal",
    labels={"taux_fake_pct": "Taux de fake (%)", "subject": "Sujet"},
    color="taux_fake_pct",
    color_continuous_scale="RdYlGn_r",
    text="taux_fake_pct"
)

fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

print("### Taux de fake par sujet principal")
for _, row in fake_rate.iterrows():
    print(f"   {row['subject']:<30} : {row['taux_fake_pct']}%")

### Taux de fake par sujet principal
   health-care                    : 35.4%
   foreign-policy                 : 33.6%
   immigration                    : 32.5%
   elections                      : 32.5%
   candidates-biography           : 31.8%
   jobs                           : 25.7%
   taxes                          : 25.1%
   state-budget                   : 25.0%
   crime                          : 24.5%
   federal-budget                 : 23.6%
   economy                        : 20.7%
   education                      : 20.0%


**Interprétation** : certains sujets politiques concentrent plus de déclarations fausses.
Cela peut s'expliquer par la nature du sujet (plus facile à exagérer ou déformer)
mais aussi par un biais de sélection de PolitiFact qui choisit les déclarations à vérifier.

**Biais de sélection** : PolitiFact ne vérifie pas au hasard — il cible les déclarations
qui semblent douteuses ou virales. Le taux de fake par sujet reflète donc
autant les habitudes de fact-checking que la réalité politique.

**Impact sur la modélisation** : si le sujet est utilisé comme feature,
le modèle risque d'associer certains thèmes à des labels de façon
spurieuse plutôt que sémantique.

## Analyse complémentaire 3 — Mots les plus fréquents par label

On analyse ici le vocabulaire utilisé selon le niveau de véracité.
Cette analyse justifie l'usage du TF-IDF comme première approche de modélisation
et permet d'identifier si certains mots sont discriminants.

In [39]:
STOPWORDS_EN = [
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "is", "are", "was", "were", "be", "been",
    "has", "have", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "that", "this", "it",
    "its", "they", "them", "their", "he", "she", "we", "our", "you",
    "your", "my", "his", "her", "not", "no", "more", "than", "as",
    "i", "s", "says", "say", "said"
]

# Groupes polaires : clairement faux vs clairement vrai
GROUPES = {
    "Clairement faux (pants-fire + false)" : ["pants-fire", "false"],
    "Nuancé (barely-true + half-true)"     : ["barely-true", "half-true"],
    "Clairement vrai (mostly-true + true)" : ["mostly-true", "true"]
}

COULEURS_GROUPES = ["#d73027", "#fdae61", "#1a9850"]

fig_bars = []

for (nom, labels), couleur in zip(GROUPES.items(), COULEURS_GROUPES):
    # Textes du groupe
    textes = train[train["label"].isin(labels)]["statement"].fillna("").tolist()
    
    # CountVectorizer : fréquence brute
    vec = CountVectorizer(
        stop_words=STOPWORDS_EN,
        max_features=500,
        ngram_range=(1, 1),
        lowercase=True,
        token_pattern=r"\b[a-zA-Z]{3,}\b"  # mots de 3 lettres minimum
    )
    X = vec.fit_transform(textes)
    
    # Top 10 mots
    freq = X.sum(axis=0).A1
    vocab = vec.get_feature_names_out()
    top_idx = np.argsort(freq)[::-1][:10]
    
    top_mots = [vocab[i] for i in top_idx]
    top_freq = [freq[i] for i in top_idx]
    
    fig_bars.append((nom, top_mots[::-1], top_freq[::-1], couleur))

# Visualisation : 3 barplots horizontaux côte à côte
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[nom for nom, _, _, _ in fig_bars]
)

for col_idx, (nom, mots, freqs, couleur) in enumerate(fig_bars, start=1):
    fig.add_trace(
        go.Bar(
            x=freqs,
            y=mots,
            orientation="h",
            marker_color=couleur,
            showlegend=False
        ),
        row=1, col=col_idx
    )

fig.update_layout(
    title_text="Top 10 mots les plus fréquents par groupe de labels",
    height=400
)
fig.show()

In [40]:
# TF-IDF : mots les PLUS DISTINCTIFS de chaque groupe (pas juste les plus fréquents)
# Un mot à score TF-IDF élevé est fréquent dans ce groupe MAIS rare dans les autres

for (nom, labels), couleur in zip(GROUPES.items(), COULEURS_GROUPES):
    textes_groupe  = train[train["label"].isin(labels)]["statement"].fillna("").tolist()
    textes_autres  = train[~train["label"].isin(labels)]["statement"].fillna("").tolist()
    
    # On entraîne le TF-IDF sur tout le corpus
    tous_textes = textes_groupe + textes_autres
    labels_bin  = [1] * len(textes_groupe) + [0] * len(textes_autres)
    
    vec = TfidfVectorizer(
        stop_words=STOPWORDS_EN,
        max_features=5000,
        ngram_range=(1, 2),
        lowercase=True,
        token_pattern=r"\b[a-zA-Z]{3,}\b"
    )
    X = vec.fit_transform(tous_textes)
    
    # Moyenne TF-IDF dans le groupe
    X_groupe = X[:len(textes_groupe)]
    mean_tfidf = X_groupe.mean(axis=0).A1
    vocab = vec.get_feature_names_out()
    
    top_idx = np.argsort(mean_tfidf)[::-1][:10]
    top_mots  = [vocab[i] for i in top_idx]
    top_score = [round(mean_tfidf[i], 4) for i in top_idx]
    
    print(f"\n### Mots distinctifs — {nom}")
    for mot, score in zip(top_mots, top_score):
        print(f"   {mot:<25} score TF-IDF : {score}")


### Mots distinctifs — Clairement faux (pants-fire + false)
   obama                     score TF-IDF : 0.0183
   president                 score TF-IDF : 0.0138
   state                     score TF-IDF : 0.0122
   health                    score TF-IDF : 0.0122
   barack                    score TF-IDF : 0.0119
   tax                       score TF-IDF : 0.0114
   care                      score TF-IDF : 0.0114
   percent                   score TF-IDF : 0.0109
   barack obama              score TF-IDF : 0.0104
   people                    score TF-IDF : 0.01

### Mots distinctifs — Nuancé (barely-true + half-true)
   percent                   score TF-IDF : 0.0174
   obama                     score TF-IDF : 0.0132
   state                     score TF-IDF : 0.0126
   tax                       score TF-IDF : 0.0111
   president                 score TF-IDF : 0.0107
   health                    score TF-IDF : 0.0106
   people                    score TF-IDF : 0.0106
   year          

**Ce qu'on observe** : certains mots et bigrammes apparaissent plus fréquemment
dans les déclarations fausses ou vraies. Cela confirme que le contenu textuel
contient un signal discriminant, même si faible.

**Limite importante** : les mots les plus fréquents ("percent", "tax", "million")
sont souvent partagés entre les groupes. Le TF-IDF seul capte les tendances
de surface mais ne comprend pas le sens — "taxes increased" et
"taxes did not increase" utilisent les mêmes mots.

**Justification du choix BERT** : c'est précisément pourquoi les embeddings
contextuels (BERT) sont recommandés pour ce dataset. BERT comprend la negation,
le contexte, et la nuance — ce que TF-IDF ne peut pas faire.

## Analyse complémentaire 4 — Distribution des labels sur valid et test

On vérifie que les 3 splits (train / valid / test) ont des distributions
de labels comparables. Un déséquilibre entre splits pourrait fausser
l'évaluation du modèle.

In [41]:
SPLITS = {"Train": train, "Valid": valid, "Test": test}

# Calcul des distributions en pourcentage pour chaque split
distributions = {}
for nom, df in SPLITS.items():
    counts = df["label"].value_counts().reindex(ORDRE_LABELS).fillna(0)
    pcts   = (counts / counts.sum() * 100).round(2)
    distributions[nom] = {"counts": counts, "pcts": pcts}

# Graphique comparatif : 3 barplots côte à côte
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=list(SPLITS.keys())
)

for col_idx, (nom, data) in enumerate(distributions.items(), start=1):
    fig.add_trace(
        go.Bar(
            x=ORDRE_LABELS,
            y=data["pcts"].values,
            marker_color=COULEURS,
            text=[f"{v:.1f}%" for v in data["pcts"].values],
            textposition="outside",
            showlegend=False
        ),
        row=1, col=col_idx
    )

fig.update_layout(
    title_text="Distribution des labels — comparaison Train / Valid / Test (%)",
    height=420
)
fig.update_yaxes(title_text="% déclarations", row=1, col=1)
fig.show()

# Résumé chiffré comparatif
print("### Distribution comparative (%)")
print(f"{'Label':<20} {'Train':>8} {'Valid':>8} {'Test':>8}")
print("-" * 50)
for label in ORDRE_LABELS:
    t = distributions["Train"]["pcts"][label]
    v = distributions["Valid"]["pcts"][label]
    s = distributions["Test"]["pcts"][label]
    print(f"{label:<20} {t:>7.1f}% {v:>7.1f}% {s:>7.1f}%")

### Distribution comparative (%)
Label                   Train    Valid     Test
--------------------------------------------------
pants-fire               8.2%     9.0%     7.3%
false                   19.5%    20.5%    19.6%
barely-true             16.1%    18.5%    16.7%
half-true               20.6%    19.3%    20.9%
mostly-true             19.2%    19.6%    19.0%
true                    16.4%    13.2%    16.4%


In [42]:
# Vérification statistique : écart max entre train et test par label
print("### Écart de distribution entre Train et Test")
ecarts = []
for label in ORDRE_LABELS:
    t = distributions["Train"]["pcts"][label]
    s = distributions["Test"]["pcts"][label]
    ecart = abs(t - s)
    ecarts.append(ecart)
    statut = "⚠️ écart notable" if ecart > 3 else "✓ OK"
    print(f"   {label:<20} : {ecart:.1f} pts  {statut}")

print(f"\n   Écart moyen : {sum(ecarts)/len(ecarts):.2f} pts")
print(f"   Écart max   : {max(ecarts):.2f} pts (label : {ORDRE_LABELS[ecarts.index(max(ecarts))]})") 

### Écart de distribution entre Train et Test
   pants-fire           : 0.9 pts  ✓ OK
   false                : 0.2 pts  ✓ OK
   barely-true          : 0.6 pts  ✓ OK
   half-true            : 0.3 pts  ✓ OK
   mostly-true          : 0.1 pts  ✓ OK
   true                 : 0.1 pts  ✓ OK

   Écart moyen : 0.36 pts
   Écart max   : 0.93 pts (label : pants-fire)


**Ce qu'on vérifie ici** : les 3 splits doivent avoir des distributions similaires
pour que l'évaluation soit fiable. Si le test set avait 2x plus de `pants-fire`
que le train set, les métriques seraient trompeuses.

**Résultat attendu** : les distributions sont proches entre les splits
(LIAR a été construit avec une stratification implicite).
Un écart < 3 points de pourcentage par classe est acceptable.

**Impact sur l'évaluation** : si les distributions sont homogènes,
l'accuracy sur le test set est directement comparable à celle sur le train.
Dans le cas contraire, il faudrait privilégier le F1-score macro
qui donne le même poids à chaque classe indépendamment de sa fréquence.